# Data Collection - Study 2b

## Load libraries

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import openai
import json
import time
import re
from tqdm import tqdm

## Prepare ChatGPT

In [ ]:
openai.api_key = os.getenv("OPENAI_API_KEY")

def analyzeGPT(
    text,
):
    prompt = text

    return openai.ChatCompletion.create(
        model='gpt-3.5-turbo',
        messages = [{"role": "user", "content": prompt}]
    )

## Prepare & fetch data

In [ ]:
df = pd.DataFrame(columns=["ai", "age", "risk-taking", "cond", "iteration", "raw"])
iterations = 10
ages = [15, 30, 50]
risk_taking = ['averse', 'neutral', 'seeking']
conditions = ['control', 'engineered']

In [ ]:
def addData(
    curAI,
    curAge,
    curRisk,
    iteration,
    cond,
    raw
):
    global df
    
    df.loc[len(df)] = {
        "ai": curAI,
        "age": curAge,
        "risk-taking": curRisk,
        "cond": cond,
        "iteration": iteration,
        "raw": raw
    }

curAI = 'ChatGPT'

for curAge in ages:
    print(f'age: {curAge}')
    for cond in conditions:
        print(f'cond: {cond}')
        risk_texts = {
            "averse" : f"Hi ChatGPT, I'm a {curAge}-year-old looking to invest $10,000. I'm not willing to take too many risks with my investment, and I'm hoping for some advice on what products I should consider investing in and how much I should allocate to each. Can you provide me with some recommendations? Please provide me with a table with the type of the investment, the name, the ticker symbol and the amount I should invest.",
            "neutral": f"Hi ChatGPT, I'm a {curAge}-year-old looking to invest $10,000. I'm willing to take some risks with my investment, and I'm hoping for some advice on what products I should consider investing in and how much I should allocate to each. Can you provide me with some recommendations? Please provide me with a table with the type of the investment, the name, the ticker symbol and the amount I should invest.",
            "seeking": f"Hi ChatGPT, I'm a {curAge}-year-old looking to invest $10,000. I'm willing to take a lot of risks with my investment, and I'm hoping for some advice on what products I should consider investing in and how much I should allocate to each. Can you provide me with some recommendations? Please provide me with a table with the type of the investment, the name, the ticker symbol and the amount I should invest."
        }
        
        if cond == 'engineered':
            risk_texts = {
            "averse" : f"Hi ChatGPT, I'm a {curAge}-year-old looking to invest $10,000. I'm not willing to take too many risks with my investment, and I'm hoping for some advice on what products I should consider investing in and how much I should allocate to each. Avoid common investment mistakes such as lack of diversification, cluster risks and active management. Can you provide me with some recommendations? Please provide me with a table with the type of the investment, the name, the ticker symbol and the amount I should invest.",
            "neutral": f"Hi ChatGPT, I'm a {curAge}-year-old looking to invest $10,000. I'm willing to take some risks with my investment, and I'm hoping for some advice on what products I should consider investing in and how much I should allocate to each. Avoid common investment mistakes such as lack of diversification, cluster risks and active management. Can you provide me with some recommendations? Please provide me with a table with the type of the investment, the name, the ticker symbol and the amount I should invest.",
            "seeking": f"Hi ChatGPT, I'm a {curAge}-year-old looking to invest $10,000. I'm willing to take a lot of risks with my investment, and I'm hoping for some advice on what products I should consider investing in and how much I should allocate to each. Avoid common investment mistakes such as lack of diversification, cluster risks and active management. Can you provide me with some recommendations? Please provide me with a table with the type of the investment, the name, the ticker symbol and the amount I should invest."
        }

        for curRisk in risk_taking:
            print(f'risk-taking: {curRisk}')
            risk_text = risk_texts[curRisk]

            for i in tqdm(range(0, iterations)):
                    res = analyzeGPT(text = risk_text)
                    response_message = res["choices"][0]["message"]
                    addData(curAI, curAge, curRisk, i, cond, response_message.content)

            

In [ ]:
df.to_csv('./data/ageRisk.csv')

## Data reparation & Investment extraction

In [ ]:
def extract_investments1(text):
    pattern = r"\|\s*(.*?)\s*\|\s*(.*?)\s*\|\s*(.*?)\s*\|\s*(.*?)\s*\|"
    
    matches = re.findall(pattern, text)[1:]
    if len(matches) > 0:
        if matches[0][0].count('-') >= 3:
            matches = matches[1:]

    investments = []
    for match in matches:
        if match[1].strip().lower() == "name" or match[1].strip().lower() == "allocation":
            return investments
        if match[0].count('-') < 3:
            investment = {
                "Investment Type": match[0].strip().replace('**',''),
                "Name": match[1].strip(),
                "Ticker Symbol": match[2].strip(),
                "Amount to Invest": match[3].strip()
            }
            investments.append(investment)
    
    return investments

def extract_investments2(text):
    pattern = r"\s*\n\s*-+\s*\|\s*-+\s*\|\s*-+\s*\|\s*-+\s*\n((?:\s*.+\s*\|\s*.+\s*\|\s*.+\s*\|\s*.+\s*\n)+)"
    matches = re.findall(pattern, text)

    investments = []
    for match in matches:
        rows = match.strip().split('\n')
        
        for row in rows:
            investment_info = re.split(r"\s*\|\s*", row.strip())
            if len(investment_info) == 4:
                investment = {
                    "Investment Type": investment_info[0].strip().replace('**',''),
                    "Name": investment_info[1],
                    "Ticker Symbol": investment_info[2],
                    "Amount to Invest": investment_info[3]
                }
                investments.append(investment)

    return investments

def extract_investments3(text):
    investments = []
    pattern = r"\s*(.*?)\s*\|\s*(.*?)\s*\|\s*(.*?)\s*\|\s*(.*?)\s*"
    matches = re.findall(pattern, text)[2:]

    for match in matches:
        investment = {
            "Investment Type": match[0].strip().replace('**',''),
            "Name": match[1].strip(),
            "Ticker Symbol": match[2].strip(),
            "Amount to Invest": match[3].strip()
        }

        investments.append(investment)

    return investments

def extract_investments(row):
    raw = row['raw']
    methods = [extract_investments1, extract_investments2, extract_investments3]
    
    for method in methods:
        investments = method(raw)
        if investments:
            return investments
    
    # Return an empty list if none of the methods successfully extract the investments
    return []

def get_correct_portfolio(row):
    investments = row['investments']
    curAge = row['age']
    curRisk = row['risk-taking']
    curCond = row['cond']
    risk_texts = {
        "averse" : f"Hi ChatGPT, I'm a {curAge}-year-old looking to invest $10,000. I'm not willing to take too many risks with my investment, and I'm hoping for some advice on what products I should consider investing in and how much I should allocate to each. Can you provide me with some recommendations? Please provide me with a table with the type of the investment, the name, the ticker symbol and the amount I should invest.",
        "neutral": f"Hi ChatGPT, I'm a {curAge}-year-old looking to invest $10,000. I'm willing to take some risks with my investment, and I'm hoping for some advice on what products I should consider investing in and how much I should allocate to each. Can you provide me with some recommendations? Please provide me with a table with the type of the investment, the name, the ticker symbol and the amount I should invest.",
        "seeking": f"Hi ChatGPT, I'm a {curAge}-year-old looking to invest $10,000. I'm willing to take a lot of risks with my investment, and I'm hoping for some advice on what products I should consider investing in and how much I should allocate to each. Can you provide me with some recommendations? Please provide me with a table with the type of the investment, the name, the ticker symbol and the amount I should invest."
    }

    if curCond == 'engineered':
        risk_texts = {
            "averse" : f"Hi ChatGPT, I'm a {curAge}-year-old looking to invest $10,000. I'm not willing to take too many risks with my investment, and I'm hoping for some advice on what products I should consider investing in and how much I should allocate to each. Avoid common investment mistakes such as lack of diversification, cluster risks and active management. Can you provide me with some recommendations? Please provide me with a table with the type of the investment, the name, the ticker symbol and the amount I should invest.",
            "neutral": f"Hi ChatGPT, I'm a {curAge}-year-old looking to invest $10,000. I'm willing to take some risks with my investment, and I'm hoping for some advice on what products I should consider investing in and how much I should allocate to each. Avoid common investment mistakes such as lack of diversification, cluster risks and active management. Can you provide me with some recommendations? Please provide me with a table with the type of the investment, the name, the ticker symbol and the amount I should invest.",
            "seeking": f"Hi ChatGPT, I'm a {curAge}-year-old looking to invest $10,000. I'm willing to take a lot of risks with my investment, and I'm hoping for some advice on what products I should consider investing in and how much I should allocate to each. Avoid common investment mistakes such as lack of diversification, cluster risks and active management. Can you provide me with some recommendations? Please provide me with a table with the type of the investment, the name, the ticker symbol and the amount I should invest."
        }
        
    risk_text = risk_texts[curRisk]
    
    while investments == []:
        res = analyzeGPT(text = risk_text)
        response_message = res["choices"][0]["message"]
        row['raw'] = response_message.content
        investments = extract_investments(row)
        
    row['investments'] = investments
    return row


In [ ]:
df_ageRisk = pd.read_csv('./data/ageRisk.csv', index_col=0)
df_ageRisk['investments'] = df_ageRisk.apply(extract_investments, axis=1)

In [ ]:
df_ageRisk.at[17, 'investments'] = [] # not deterministic - provides examples
df_ageRisk.at[21, 'investments'] = [] # not deterministic - provides examples
df_ageRisk.at[35, 'investments'] = [] # wrong format
df_ageRisk.at[40, 'investments'] = [] # not deterministic - provides examples ABC, XYZ
df_ageRisk.at[41, 'investments'] = [] # not deterministic - provides examples FB, APPL, AMZN, NFLX, GOOGL
df_ageRisk.at[50, 'investments'] = [] # not deterministic - wrong tickers
df_ageRisk.at[51, 'investments'] = [] # not deterministic - provides general descriptions
df_ageRisk.at[54, 'investments'] = [] # not deterministic - missing tickers
df_ageRisk.at[58, 'investments'] = [] # not deterministic - provides general descriptions
df_ageRisk.at[64, 'investments'] = [] # not deterministic - examples VTSMX / VTI
df_ageRisk.at[67, 'investments'] = [] # not deterministic - wrong tickers ABC etc.
df_ageRisk.at[101, 'investments'] = [] # not deterministic - missing tickers
df_ageRisk.at[114, 'investments'] = [] # wrong format
df_ageRisk.at[121, 'investments'] = [] # not deterministic - missing tickers
df_ageRisk.at[131, 'investments'] = [] # not deterministic - missing tickers
df_ageRisk.at[135, 'investments'] = [] # not deterministic - missing tickers
df_ageRisk.at[142, 'investments'] = [] # not deterministic - wrong tickers REITs
df_ageRisk.at[144, 'investments'] = [] # not deterministic - wrong tickers ABC etc.
df_ageRisk.at[146, 'investments'] = [] # not deterministic - provides examples
df_ageRisk.at[147, 'investments'] = [] # amount ranges
df_ageRisk.at[148, 'investments'] = [] # not deterministic - wrong tickers
df_ageRisk.at[149, 'investments'] = [] # not deterministic - missing & wrong tickers 
df_ageRisk.at[173, 'investments'] = [] 
df_ageRisk.at[174, 'investments'] = [] # not deterministic - missing tickers
df_ageRisk.at[176, 'investments'] = [] # not deterministic - missing amount

In [ ]:
df_ageRisk_complete = df_ageRisk.apply(get_correct_portfolio, axis=1)
df_ageRisk_complete.to_csv('./data/ageRisk_complete.csv')